In [1]:
from nltk import CFG, ChartParser

# Define Grammar and Parser

In [2]:
prolog_grammar = CFG.fromstring("""
E -> P C
P -> 'gamma' '(' A ')' '.' P |
A -> | AA AAA
AAA -> | ',' AA AAA
AA -> 'alpha' | '['  T ']'
T -> | '[' T ']' TTT | 'alpha' TT
TT -> '-' AA | ',' T |
TTT -> ',' T |                                
C -> | 'mu' '(' W ')' ':-' B '.' C 
W -> | V 
V -> VV VVV 
VVV -> | ',' VV VVV
VV -> 'nu'
B -> BB BBB 
BBB -> ',' BB BBB |
BB -> BBBB  BBBBB 
BBBBB -> ';' BBBB BBBBB |
BBBB -> 'gamma' '(' A ')' | '!'
""")

prolog_parser = ChartParser(prolog_grammar)

In [3]:
working_sentences = [
"gamma ( ) .",
"gamma ( alpha ) .",
"gamma ( alpha , alpha , alpha ) .",
"gamma ( [ ] , alpha ) .",
"gamma ( alpha , [ ] ) .",
"gamma ( alpha , [ [ ] ] ) .",
"gamma ( alpha , [ alpha - alpha ] ) .",
"gamma ( alpha , [ alpha - [ alpha ] ] ) .",
"gamma ( alpha , [ alpha , alpha , alpha ] ) .",
"gamma ( alpha , [ alpha , [ alpha , alpha ] , alpha ] ) .",
"gamma ( alpha ) . gamma ( alpha ) .",
"mu ( nu , nu ) :- gamma ( alpha ) , gamma ( alpha ) . mu ( ) :- gamma ( ) ; gamma ( ) , ! .  ",
]

failing_sentences = [
"gamma alpha",
"gamma ( alpha alpha ) .",
"gamma ( alpha , alpha )",
"gamma ( alpha , [ [ alpha ] - alpha ] ) .",
"mu ( nu , nu ) :- gamma ( alpha )",
"mu ( nu  nu ) :- gamma ( alpha ) .",
"mu ( nu , nu ) :- gamma ( alpha ) mu ( nu , nu ) :- gamma ( alpha )",
"gamma ( alpha , alpha ) mu ( nu , nu ) :- gamma ( alpha ) . ",
]

## Debug Code for Grammar

In [52]:
# for sentence in working_sentences:
tree = prolog_parser.chart_parse(working_sentences[-3].split(), trace=1)


|.g.(.a.,.[.a.,.[.a.,.a.].,.a.].)...|
|[-] . . . . . . . . . . . . . . . .| [0:1] 'gamma'
|. [-] . . . . . . . . . . . . . . .| [1:2] '('
|. . [-] . . . . . . . . . . . . . .| [2:3] 'alpha'
|. . . [-] . . . . . . . . . . . . .| [3:4] ','
|. . . . [-] . . . . . . . . . . . .| [4:5] '['
|. . . . . [-] . . . . . . . . . . .| [5:6] 'alpha'
|. . . . . . [-] . . . . . . . . . .| [6:7] ','
|. . . . . . . [-] . . . . . . . . .| [7:8] '['
|. . . . . . . . [-] . . . . . . . .| [8:9] 'alpha'
|. . . . . . . . . [-] . . . . . . .| [9:10] ','
|. . . . . . . . . . [-] . . . . . .| [10:11] 'alpha'
|. . . . . . . . . . . [-] . . . . .| [11:12] ']'
|. . . . . . . . . . . . [-] . . . .| [12:13] ','
|. . . . . . . . . . . . . [-] . . .| [13:14] 'alpha'
|. . . . . . . . . . . . . . [-] . .| [14:15] ']'
|. . . . . . . . . . . . . . . [-] .| [15:16] ')'
|. . . . . . . . . . . . . . . . [-]| [16:17] '.'
|# . . . . . . . . . . . . . . . . .| [0:0] P  -> *
|. # . . . . . . . . . . . . . . . .| [1:1] P  -> *
|. 

In [32]:
trees = prolog_parser.parse(working_sentences[2].split())
for t in trees:
    print(t)

# Test Working Sentences

In [4]:
for sentence in working_sentences:
    trees = prolog_parser.parse(sentence.split())
    trees_list = [t for t in trees]

    assert len(trees_list) == 1

    print(f"The sentence \033[1m{sentence}\033[0m has {len(trees_list)} possible tree")
    for tree in trees_list:
        tree.pretty_print()

The sentence gamma ( ) . has 1 possible tree
               E             
            ___|___________   
           P               | 
   ________|___________    |  
  |    |   |   |   A   P   C 
  |    |   |   |   |   |   |  
gamma  (   )   .  ... ... ...

The sentence gamma ( alpha ) . has 1 possible tree
                    E                  
                ____|________________   
               P                     | 
   ____________|_________________    |  
  |    |   |   |         A       |   | 
  |    |   |   |     ____|___    |   |  
  |    |   |   |    AA      AAA  P   C 
  |    |   |   |    |        |   |   |  
gamma  (   )   .  alpha     ... ... ...

The sentence gamma ( alpha , alpha , alpha ) . has 1 possible tree
                              E                        
                          ____|______________________   
                         P                           | 
   ______________________|_______________________    |  
  |    |   |   |              A 

# Test Failing Sentences

In [61]:
for sentence in failing_sentences:
    trees = prolog_parser.parse(sentence.split())
    trees_list = [t for t in trees]

    assert len(trees_list) == 0

    print(f"The sentence \033[1m{sentence}\033[0m has {len(trees_list)} amount of possible trees")
    for tree in trees_list:
        tree.pretty_print()

The sentence gamma alpha has 0 amount of possible trees
The sentence gamma ( alpha alpha ) . has 0 amount of possible trees
The sentence gamma ( alpha , alpha ) has 0 amount of possible trees
The sentence gamma ( alpha , [ [ alpha ] - alpha ] ) . has 0 amount of possible trees
The sentence mu ( nu , nu ) :- gamma ( alpha ) has 0 amount of possible trees
The sentence mu ( nu  nu ) :- gamma ( alpha ) . has 0 amount of possible trees
The sentence mu ( nu , nu ) :- gamma ( alpha ) mu ( nu , nu ) :- gamma ( alpha ) has 0 amount of possible trees
The sentence gamma ( alpha , alpha ) mu ( nu , nu ) :- gamma ( alpha ) .  has 0 amount of possible trees
